# Hello, world: 첫 CuTe DSL 커널

**CuTe DSL**은 GPU 커널을 파이썬으로 작성해서 곧바로 CUDA로 컴파일합니다. 데코레이터 두 개가 이 일을
합니다 — **`@cute.kernel`**은 GPU 스레드 하나하나가 실행하는 *디바이스* 코드를 표시하고,
**`@cute.jit`**은 그 커널을 실행시키는 *호스트* 함수를 표시합니다. 커널에는 일반 PyTorch CUDA
텐서를 그대로 넘기고, 결과도 그대로 돌려받습니다 — C++도, 별도 빌드 단계도 없습니다.

이번 1장은 일부러 아주 작게 잡았습니다: 배열 두 개를 더하고 GPU에서 인사말을 출력하는 커널
하나뿐입니다. 이후 모든 장이 이 형태 위에 쌓입니다.

원문: [`NVIDIA/cutlass` examples/python/CuTeDSL/notebooks/1_dsl_features/01_hello_world.ipynb](https://github.com/NVIDIA/cutlass/blob/098de2a652cf8f00fd70b2df54051c7eccbb855a/examples/python/CuTeDSL/notebooks/1_dsl_features/01_hello_world.ipynb) (BSD-3-Clause, commit `098de2a` 기준)

**배울 내용:** 데코레이터 두 개 — `@cute.kernel`(디바이스) / `@cute.jit`(호스트 런처); 스레드가
`cute.arch.thread_idx()` / `block_idx()` / `block_dim()`으로 자기 위치를 찾는 법; `cute.printf`로
디바이스에서 출력하는 법; `cute.runtime.from_dlpack`을 통한 cutlass ↔ PyTorch 왕복.

**실행 환경:** 공식 노트북 원문 기준 CUDA GPU면 어느 것이든 동작합니다(단, `Arch` enum이
`sm_80`부터 시작해 T4는 해당 없음 — README 참고). 이 튜토리얼은 **Colab L4**에서 실제로
실행해 검증했습니다.

## 0. 환경 확인

재현성을 위해 먼저 GPU 정보를 남깁니다.

In [ ]:
!nvidia-smi

## 0-1. CuTe DSL 설치

`nvidia-cutlass-dsl` 패키지가 CuTe DSL 본체입니다. 버전은 `4.8.0`으로 고정합니다(아래 함정
설명이 이 버전의 패키징 구조에 묶여 있음). 기본 설치는 CUDA 12.x용(`nvidia-cutlass-dsl-libs-cu12`)
이고, `cu13` extra를 주면 CUDA 13.x용(`nvidia-cutlass-dsl-libs-cu13`)이 대신 설치됩니다(PyPI
metadata로 확인). **이 노트북은 Colab의 CUDA 13.0 환경에서도 기본(cu12) 설치로 문제없이
검증했습니다** — nvidia-smi가 보여주는 CUDA 버전(드라이버가 지원하는 최대 버전)과 패키지가
요구하는 CUDA 버전이 반드시 같을 필요는 없습니다.

In [ ]:
!pip install -q nvidia-cutlass-dsl==4.8.0

### 함정: 같은 커널에서 설치 직후 import하면 실패합니다

`nvidia-cutlass-dsl`은 실제 코드를
`/usr/local/lib/python3.13/dist-packages/nvidia_cutlass_dsl/dsl_packages/`에 넣고 `.pth`
파일로 그 경로를 `sys.path`에 등록하는 방식입니다. 문제는 `.pth`가 **인터프리터(커널) 시작
시점**에 한 번만 읽힌다는 것 — 커널이 이미 떠 있는 상태에서 `pip install`을 돌리면 `.pth`가
새로 생겨도 지금 떠 있는 커널의 `sys.path`엔 반영되지 않습니다.

그 결과 바로 다음 셀에서 `import cutlass`를 하면 `ModuleNotFoundError: No module named
'cutlass'`가 납니다 — 패키지는 분명히 설치됐는데도입니다. (별도 `python3` 프로세스로
`import cutlass`를 하면 성공하는데, 그건 새 프로세스라 `.pth`를 다시 읽기 때문입니다.)

해결: 커널을 재시작하지 않고 `site.main()`으로 `.pth`를 다시 읽게 하면 됩니다.

In [ ]:
import site
site.main()  # 방금 설치된 nvidia-cutlass-dsl의 .pth 경로를 sys.path에 반영

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. 커널 — `@cute.kernel`

`@cute.kernel` 함수는 **GPU 스레드 하나**가 실행하는 코드입니다. GPU는 이 코드를 수천 개 스레드가
동시에 실행합니다. 스레드는 내장 함수 세 개로 자기 위치를 찾습니다 — `cute.arch.thread_idx()`(블록
안에서의 순번), `cute.arch.block_idx()`(어느 블록인지), `cute.arch.block_dim()`(블록당 스레드 수) —
각각 3-튜플 `(x, y, z)`이고 여기선 `x`만 씁니다. 전역 원소 인덱스는 `block * block_size + thread`입니다.

커널 인자는 **`cutlass.Array`** — GPU 메모리를 가리키는 타입 핸들이고 파이썬 시퀀스처럼 인덱싱합니다.
`cute.printf`는 디바이스에서 출력하는 함수 — `{}` 자리표시자가 있는 포맷 문자열 뒤에 값을 넘깁니다.
여기선 스레드 0만 인사말을 찍습니다. 수천 개 스레드가 다 찍으면 콘솔이 넘칩니다.

In [ ]:
@cute.kernel
def hello_kernel(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32):
    tx, _, _ = cute.arch.thread_idx()   # lane within the block
    bx, _, _ = cute.arch.block_idx()    # which block
    bdx, _, _ = cute.arch.block_dim()   # threads per block
    i = bx * bdx + tx                   # this thread's global element

    # One thread says hello (every thread printing would flood the console).
    if i == 0:
        cute.printf("hello from the GPU -- block {}, thread {}\n", bx, tx)

    # Each thread adds one element. Guard the tail: the grid rounds up to whole
    # blocks, so the last block can have threads whose index runs past the array.
    if i < N:
        c[i] = a[i] + b[i]

## 2. 런처 — `@cute.jit`

`@cute.jit` 호스트 함수가 실행 모양을 정하고 `.launch(grid=, block=)`으로 커널을 시작합니다 —
`grid`는 블록 개수, `block`은 블록당 스레드 수. 여기선 원소당 스레드 1개를 쓰고, 전체 원소를 다
덮도록 grid를 올림합니다.

호출할 땐 각 PyTorch CUDA 텐서를 `cute.runtime.from_dlpack`으로 감쌉니다 — `cutlass.Array`가
필요한 자리 어디서든 쓸 수 있는 제로카피 뷰입니다 — 그다음 호스트 함수를 그냥 부르면 됩니다.

In [ ]:
@cute.jit
def hello(a: cutlass.Array, b: cutlass.Array, c: cutlass.Array, N: cutlass.Int32):
    block = (256, 1, 1)
    grid = ((N + 255) // 256, 1, 1)   # one thread per element, rounded up to whole blocks
    hello_kernel(a, b, c, N).launch(grid=grid, block=block)


N = 1024
a = torch.randn(N, dtype=torch.float32, device="cuda")
b = torch.randn(N, dtype=torch.float32, device="cuda")
c = torch.zeros(N, dtype=torch.float32, device="cuda")

# from_dlpack wraps each CUDA tensor as a cutlass.Array with no copy.
hello(cute.runtime.from_dlpack(a), cute.runtime.from_dlpack(b), cute.runtime.from_dlpack(c), N)

torch.testing.assert_close(c.cpu(), (a + b).cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output (the GPU greeting prints once, from thread 0):
# hello from the GPU -- block 0, thread 0
# PASS

### 실측 검증 (2026-09-27, Colab L4)

이 노트북 전체를 `.ipynb` 단위로(스크립트가 아니라 실제 셀 순서대로 하나의 커널에서) Colab
L4(23GB, driver 580.82.07, CUDA 13.0)에서 실행해 확인했습니다.

- `site.main()` 셀 없이 실행하면 바로 위 함정대로 `ModuleNotFoundError`가 재현됩니다.
- `site.main()` 셀을 넣으면 `import cutlass`부터 마지막 셀까지 에러 없이 통과합니다.

**원문 "Expected output" 주석과 다른 점 하나: `cute.printf` 출력이 셀에 기본적으로 안 뜹니다.**
위 셀을 그대로 실행하면 실제 셀 출력은 `PASS`뿐이고, `hello from the GPU -- block 0, thread 0`
줄은 나타나지 않습니다. `assert_close`는 통과하니 커널 자체는 정상 동작 — 안 보인 건 그 출력
한 줄뿐입니다.

**관찰한 사실(이 환경, 이 세션 한정):**
- 위 셀만 실행하면 그 줄이 안 뜹니다.
- `os.dup2`로 fd 1(표준출력 파일디스크립터)을 파일에 연결한 다음 커널을 실행하고, fd를
  되돌린 뒤 그 파일 내용을 다시 `print()`하면 그 줄이 잡힙니다 — 아래 "printf 출력 보기" 셀
  참고.
- 정확한 원인(이 Colab 커널이 fd 1 직접 쓰기를 셀 출력에 왜 전달하지 않는지)은 확인하지
  못했습니다 — device printf 경로, 버퍼링, ipykernel 캡처 방식 중 어느 것이 결정적인지 이
  세션에서 가려내지 못했습니다. 로컬 Jupyter 등 다른 환경에서는 다를 수 있습니다.

커널 코드 자체는 원본과 다른 점이 없습니다. 이 저장소가 추가한 건 `site.main()` 셀, 아래
printf 캡처 헬퍼, 이 실측 관찰뿐입니다.

### printf 출력 보기 (이 저장소가 추가한 헬퍼)

`cute.printf`가 쓰는 fd 1을 임시로 파일에 연결했다가 그 내용을 다시 `print()`하는
컨텍스트매니저입니다. 이후 장에서도 device printf 결과를 셀에서 보고 싶을 때 재사용하세요.

In [ ]:
import os
import sys
import ctypes
import tempfile
import contextlib


@contextlib.contextmanager
def show_device_printf():
    """cute.printf가 fd 1에 직접 쓰는 내용을 캡처해서 with 블록이 끝난 뒤 print합니다."""
    sys.stdout.flush()
    saved_fd1 = os.dup(1)
    tmp = tempfile.NamedTemporaryFile(delete=False)
    tmp_path = tmp.name
    tmp.close()
    tmp_fd = os.open(tmp_path, os.O_WRONLY)
    os.dup2(tmp_fd, 1)
    try:
        yield
        cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
        ctypes.CDLL(None).fflush(None)
    finally:
        sys.stdout.flush()
        os.dup2(saved_fd1, 1)
        os.close(saved_fd1)
        os.close(tmp_fd)
    with open(tmp_path, "r") as f:
        captured = f.read()
    os.unlink(tmp_path)
    if captured:
        print(captured, end="")


c2 = torch.zeros(N, dtype=torch.float32, device="cuda")
with show_device_printf():
    hello(cute.runtime.from_dlpack(a), cute.runtime.from_dlpack(b), cute.runtime.from_dlpack(c2), N)

## 직접 해보기

- `N`을 바꿔서 256의 배수가 아닐 때 tail guard `if i < N`이 왜 필요한지 확인해 보세요.
- 스레드 **전부**가 자기 인덱스를 찍게 바꿔보고, 왜 인사말을 스레드 0으로만 제한했는지 체감해 보세요.
- `+`를 `*`로 바꾸거나, 세 번째 입력을 추가해 보세요.

다음 `02_control_flow`는 반복문과 컴파일타임 분기를 다루고, `2_primitives`의 `01_array_concepts`는
`cutlass.Array` 자체 — 메모리 공간, 다차원 인덱싱, 벡터화 슬라이스 — 를 더 깊이 다룹니다.
(`02_control_flow`는 이 저장소에 있고, `2_primitives/01_array_concepts`는 아직 번역하지 않았습니다.)